In [1]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery

In [2]:
load_dotenv()

PROJECT_ID = os.environ["GCP_PROJECT_ID"]

client = bigquery.Client(project=PROJECT_ID)

Python(91096) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


#### Below queries are to test and understand data pull

In [3]:
### Simple query to check data pull

query = """
SELECT
  created_at,
  type,
  repo.name AS repo_name,
  actor.login AS actor_login
FROM `githubarchive.day.20260101`
WHERE repo.name IN ('apache/spark', 'kubernetes/kubernetes')
LIMIT 200
"""


In [4]:
df = client.query(query).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
df.head()

,created_at,type,repo_name,actor_login
0,2026-01-01 11:49:23+00:00,WatchEvent,apache/spark,tarsislimadev
1,2026-01-01 11:31:47+00:00,IssueCommentEvent,kubernetes/kubernetes,bart0sh
2,2026-01-01 11:21:05+00:00,IssueCommentEvent,kubernetes/kubernetes,k8s-ci-robot
3,2026-01-01 11:27:04+00:00,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot
4,2026-01-01 11:27:04+00:00,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot


In [6]:


# df = client.query(query).to_dataframe()

print(df.head(20))
print("rows:", len(df))
print("columns:", list(df.columns))

os.makedirs("data/sample", exist_ok=True)
df.to_parquet("data/sample/tiny_githubarchive_sample.parquet", index=False)
print("saved: data/sample/tiny_githubarchive_sample.parquet")


                  created_at                           type  \
0  2026-01-01 04:33:15+00:00                    IssuesEvent   
1  2026-01-01 04:45:30+00:00  PullRequestReviewCommentEvent   
2  2026-01-01 04:45:31+00:00         PullRequestReviewEvent   
3  2026-01-01 04:33:11+00:00              IssueCommentEvent   
4  2026-01-01 04:35:02+00:00  PullRequestReviewCommentEvent   
5  2026-01-01 04:56:14+00:00                      ForkEvent   
6  2026-01-01 01:21:21+00:00                     WatchEvent   
7  2026-01-01 16:49:12+00:00              IssueCommentEvent   
8  2026-01-01 16:39:03+00:00                     WatchEvent   
9  2026-01-01 16:54:47+00:00                      ForkEvent   
10 2026-01-01 23:03:05+00:00               PullRequestEvent   
11 2026-01-01 23:03:40+00:00              IssueCommentEvent   
12 2026-01-01 02:40:25+00:00                     WatchEvent   
13 2026-01-01 17:53:23+00:00                     WatchEvent   
14 2026-01-01 17:44:15+00:00         PullRequestReviewE

In [8]:
query1 = """SELECT *
        FROM `githubarchive.day.20260101`
        LIMIT 1;"""

In [9]:
df1 = client.query(query1).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
df1

,type,public,payload,repo,actor,org,created_at,id,other
0,PushEvent,True,"{""repository_id"":1103832160,""push_id"":29415502...","{'id': 1103832160, 'name': 'olleolleolle/activ...","{'id': 211, 'login': 'olleolleolle', 'gravatar...",None,2026-01-01 14:43:00+00:00,7120780175,"{""actor"":{""display_login"":""olleolleolle""}}"


In [ ]:
### Sample query

query2 = """SELECT
        created_at,
        DATE(created_at) AS event_date,
        type AS event_type,
        repo.name AS repo_name,
        actor.login AS actor_login,
        TO_JSON_STRING(payload) AS payload_json
        FROM `githubarchive.day.20260101`
        WHERE repo.name IN ('apache/spark', 'kubernetes/kubernetes')
        AND type IN (
            'PushEvent',
            'PullRequestEvent',
            'PullRequestReviewEvent',
            'IssuesEvent'
        )
        LIMIT 200;"""


In [12]:
df2 = client.query(query2).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [15]:
df2.iloc[0]['payload_json']

'"{\\"action\\":\\"labeled\\",\\"number\\":135993,\\"pull_request\\":{\\"url\\":\\"https://api.github.com/repos/kubernetes/kubernetes/pulls/135993\\",\\"id\\":3141117146,\\"number\\":135993,\\"head\\":{\\"ref\\":\\"fix-deployment-antiaffinity-deadlock\\",\\"sha\\":\\"ba74c3eebcda00cb211bec384ab445c7e7620ac5\\",\\"repo\\":{\\"id\\":1126332762,\\"url\\":\\"https://api.github.com/repos/madzem/kubernetes\\",\\"name\\":\\"kubernetes\\"}},\\"base\\":{\\"ref\\":\\"master\\",\\"sha\\":\\"bf0e7b694f3a7355c200241825b6688ad391fae8\\",\\"repo\\":{\\"id\\":20580498,\\"url\\":\\"https://api.github.com/repos/kubernetes/kubernetes\\",\\"name\\":\\"kubernetes\\"}}},\\"label\\":{\\"id\\":1322574295,\\"node_id\\":\\"MDU6TGFiZWwxMzIyNTc0Mjk1\\",\\"url\\":\\"https://api.github.com/repos/kubernetes/kubernetes/labels/do-not-merge/invalid-commit-message\\",\\"name\\":\\"do-not-merge/invalid-commit-message\\",\\"color\\":\\"e11d21\\",\\"default\\":false,\\"description\\":\\"Indicates that a PR should not merge

In [16]:
df2

,created_at,event_date,event_type,repo_name,actor_login,payload_json
0,2026-01-01 20:04:26+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""number\"":135993,\""p..."
1,2026-01-01 20:04:30+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""number\"":135993,\""p..."
2,2026-01-01 20:04:42+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""number\"":135993,\""p..."
3,2026-01-01 20:04:47+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""unlabeled\"",\""number\"":135993,\..."
4,2026-01-01 20:30:08+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""unlabeled\"",\""number\"":135993,\..."
5,2026-01-01 20:30:10+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""unlabeled\"",\""number\"":135993,\..."
6,2026-01-01 20:37:16+00:00,2026-01-01,IssuesEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""issue\"":{\""url\"":\""..."
7,2026-01-01 10:03:27+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""unlabeled\"",\""number\"":135977,\..."
8,2026-01-01 09:38:25+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""number\"":135991,\""p..."
9,2026-01-01 09:38:25+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,"""{\""action\"":\""labeled\"",\""number\"":135991,\""p..."


In [17]:
df2['event_type'].unique()

array(['PullRequestEvent', 'IssuesEvent', 'PullRequestReviewEvent'],
      dtype=object)

In [18]:
query3 = """SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  -- Common extracted fields
  JSON_VALUE(payload, '$.action') AS action,

  -- Pull request specific
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS pr_number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,

  -- Review specific
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.day.20260101`
WHERE repo.name IN ('apache/spark', 'kubernetes/kubernetes')
  AND type IN (
    'PushEvent',
    'PullRequestEvent',
    'PullRequestReviewEvent',
    'IssuesEvent'
  )
LIMIT 300;
"""

In [20]:
df3 = client.query(query3).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [22]:
df3.head(20)

,created_at,event_date,event_type,repo_name,actor_login,action,pr_number,pr_merged,review_state
0,2026-01-01 04:33:15+00:00,2026-01-01,IssuesEvent,kubernetes/kubernetes,k8s-ci-robot,unlabeled,<NA>,<NA>,None
1,2026-01-01 04:45:31+00:00,2026-01-01,PullRequestReviewEvent,kubernetes/kubernetes,toVersus,created,<NA>,<NA>,commented
2,2026-01-01 10:03:27+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,unlabeled,135977,<NA>,None
3,2026-01-01 03:01:49+00:00,2026-01-01,PullRequestReviewEvent,kubernetes/kubernetes,utam0k,created,<NA>,<NA>,commented
4,2026-01-01 17:44:15+00:00,2026-01-01,PullRequestReviewEvent,kubernetes/kubernetes,pohly,created,<NA>,<NA>,commented
5,2026-01-01 17:39:56+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,labeled,135820,<NA>,None
6,2026-01-01 17:54:08+00:00,2026-01-01,PullRequestReviewEvent,kubernetes/kubernetes,lmktfy,created,<NA>,<NA>,commented
7,2026-01-01 15:05:10+00:00,2026-01-01,IssuesEvent,kubernetes/kubernetes,k8s-ci-robot,unlabeled,<NA>,<NA>,None
8,2026-01-01 15:18:56+00:00,2026-01-01,IssuesEvent,kubernetes/kubernetes,k8s-ci-robot,unlabeled,<NA>,<NA>,None
9,2026-01-01 15:34:12+00:00,2026-01-01,PullRequestEvent,kubernetes/kubernetes,k8s-ci-robot,closed,129247,<NA>,None


In [ ]:
### Sample data from 5 repositories - for 1 day

query4 = """SELECT
    created_at,
    DATE(created_at) AS event_date,
    type AS event_type,
    repo.name AS repo_name,
    actor.login AS actor_login,

    JSON_VALUE(payload, '$.action') AS action,

    SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS pr_number,
    SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,

    JSON_VALUE(payload, '$.review.state') AS review_state

    FROM `githubarchive.day.20260206`
    WHERE repo.name IN ('apache/spark', 'kubernetes/kubernetes', 'apache/airflow', 'tensorflow/tensorflow', 'prometheus/prometheus')
    AND (
        type = 'PushEvent'
        OR type = 'PullRequestReviewEvent'
        OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened', 'closed', 'reopened'))
        OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened', 'closed', 'reopened'))
    );
    -- LIMIT 300;
"""

In [39]:
df4 = client.query(query4).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [40]:
df4.shape

(303, 9)

In [30]:
df4.head(10)

,created_at,event_date,event_type,repo_name,actor_login,action,pr_number,pr_merged,review_state
0,2026-02-07 19:12:45+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,darshansreenivas,created,<NA>,<NA>,changes_requested
1,2026-02-07 04:35:13+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,hoteye,updated,<NA>,<NA>,commented
2,2026-02-07 04:35:13+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,hoteye,created,<NA>,<NA>,commented
3,2026-02-07 07:15:14+00:00,2026-02-07,PullRequestEvent,apache/spark,dongjoon-hyun,opened,54196,<NA>,None
4,2026-02-07 07:48:50+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,bart0sh,created,<NA>,<NA>,commented
5,2026-02-07 07:49:42+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,bart0sh,created,<NA>,<NA>,commented
6,2026-02-07 07:51:20+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,bart0sh,created,<NA>,<NA>,commented
7,2026-02-07 07:54:52+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,bart0sh,created,<NA>,<NA>,commented
8,2026-02-07 07:55:53+00:00,2026-02-07,PullRequestReviewEvent,kubernetes/kubernetes,bart0sh,created,<NA>,<NA>,commented
9,2026-02-07 14:22:24+00:00,2026-02-07,PullRequestEvent,kubernetes/kubernetes,dims,opened,136820,<NA>,None


In [31]:
df4['event_type'].unique()

array(['PullRequestReviewEvent', 'PullRequestEvent', 'PushEvent',
       'IssuesEvent'], dtype=object)

In [ ]:
### 1 month complete data from 5 repositories

query5 = """
SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,

  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,

  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202601`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened', 'closed', 'reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened', 'closed', 'reopened'))
)
ORDER BY created_at;
"""


In [48]:

df5 = client.query(query5).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [49]:
df5.shape

(8049, 9)

In [50]:
df5.head(20)

,created_at,event_date,event_type,repo_name,actor_login,action,number,pr_merged,review_state
0,2026-01-01 00:48:41+00:00,2026-01-01,PullRequestReviewEvent,apache/spark,anishshri-db,created,<NA>,<NA>,commented
1,2026-01-01 00:49:51+00:00,2026-01-01,PullRequestReviewEvent,apache/spark,anishshri-db,created,<NA>,<NA>,commented
2,2026-01-01 00:55:36+00:00,2026-01-01,PullRequestReviewEvent,apache/spark,anishshri-db,created,<NA>,<NA>,commented
3,2026-01-01 01:13:49+00:00,2026-01-01,PushEvent,apache/airflow,Lee-W,None,<NA>,<NA>,None
4,2026-01-01 01:13:49+00:00,2026-01-01,IssuesEvent,apache/airflow,Lee-W,closed,<NA>,<NA>,None
5,2026-01-01 02:16:33+00:00,2026-01-01,IssuesEvent,tensorflow/tensorflow,github-actions[bot],closed,<NA>,<NA>,None
6,2026-01-01 02:34:31+00:00,2026-01-01,PullRequestEvent,apache/airflow,ftakelait,opened,60009,<NA>,None
7,2026-01-01 02:41:33+00:00,2026-01-01,PullRequestReviewEvent,apache/airflow,vatsrahul1001,created,<NA>,<NA>,approved
8,2026-01-01 02:43:58+00:00,2026-01-01,PullRequestReviewEvent,apache/airflow,amoghrajesh,created,<NA>,<NA>,approved
9,2026-01-01 02:44:08+00:00,2026-01-01,PushEvent,apache/airflow,amoghrajesh,None,<NA>,<NA>,None


In [ ]:
### 3 months complete data for 5 repositories 

query6 = """
SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202511`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
)

UNION ALL

SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202512`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
)

UNION ALL

SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202601`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
);
"""


In [ ]:
# CREATE OR REPLACE TABLE `composed-task-345810.devinsight.scoped_events` AS

In [60]:
df6 = client.query(query6).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [61]:
df6.shape

(24323, 9)

In [62]:
### same query as above - saving the data in BigQuery for further analysis

query6 = """
CREATE OR REPLACE TABLE `composed-task-345810.devinsight.scoped_events` AS

SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202511`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
)

UNION ALL

SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202512`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
)

UNION ALL

SELECT
  created_at,
  DATE(created_at) AS event_date,
  type AS event_type,
  repo.name AS repo_name,
  actor.login AS actor_login,

  JSON_VALUE(payload, '$.action') AS action,
  SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
  SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
  JSON_VALUE(payload, '$.review.state') AS review_state

FROM `githubarchive.month.202601`
WHERE repo.name IN (
  'apache/spark',
  'kubernetes/kubernetes',
  'apache/airflow',
  'tensorflow/tensorflow',
  'prometheus/prometheus'
)
AND (
  type = 'PushEvent'
  OR type = 'PullRequestReviewEvent'
  OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
  OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
);
"""


In [63]:
client.query(query6).result()


In [ ]:
## Testing

client.query(
    "SELECT COUNT(*) FROM `composed-task-345810.devinsight.scoped_events`"
).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
project_id: str = "composed-task-345810"
dataset_id: str = "devinsight"
table_id: str = "scoped_events"

In [6]:
query7 = f"""
    
    SELECT
      created_at,
      DATE(created_at) AS event_date,
      type AS event_type,
      repo.name AS repo_name,
      actor.login AS actor_login,
      JSON_VALUE(payload, '$.action') AS action,
      SAFE_CAST(JSON_VALUE(payload, '$.number') AS INT64) AS number,
      SAFE_CAST(JSON_VALUE(payload, '$.pull_request.merged') AS BOOL) AS pr_merged,
      JSON_VALUE(payload, '$.review.state') AS review_state
    FROM `githubarchive.month.*`
    WHERE _TABLE_SUFFIX IN ('202511','202512','202601')
    AND repo.name IN (
      'apache/spark',
      'kubernetes/kubernetes',
      'apache/airflow',
      'tensorflow/tensorflow',
      'prometheus/prometheus'
    )
    AND (
      type = 'PushEvent'
      OR type = 'PullRequestReviewEvent'
      OR (type = 'PullRequestEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
      OR (type = 'IssuesEvent' AND JSON_VALUE(payload, '$.action') IN ('opened','closed','reopened'))
    );
    """

In [7]:
df7 = client.query(query7).to_dataframe()

/Users/pranavr/Desktop/Pranav/NEU /Spring26/Capstone/Proj_Code/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
df7.tail(50)

,created_at,event_date,event_type,repo_name,actor_login,action,number,pr_merged,review_state
24273,2026-01-29 18:00:56+00:00,2026-01-29,PushEvent,apache/airflow,bbovenzi,None,<NA>,<NA>,None
24274,2026-01-29 15:37:09+00:00,2026-01-29,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24275,2026-01-29 08:15:07+00:00,2026-01-29,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24276,2026-01-29 16:47:38+00:00,2026-01-29,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24277,2025-11-28 13:25:47+00:00,2025-11-28,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24278,2025-11-28 22:05:46+00:00,2025-11-28,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24279,2025-11-28 11:56:54+00:00,2025-11-28,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None
24280,2025-11-28 17:17:26+00:00,2025-11-28,PushEvent,apache/airflow,potiuk,None,<NA>,<NA>,None
24281,2025-11-28 12:37:23+00:00,2025-11-28,PushEvent,apache/spark,cloud-fan,None,<NA>,<NA>,None
24282,2025-11-28 10:12:30+00:00,2025-11-28,PushEvent,tensorflow/tensorflow,copybara-service[bot],None,<NA>,<NA>,None


In [11]:
df7['review_state'].unique()

array([None, 'changes_requested', 'commented', 'approved', 'dismissed'],
      dtype=object)